In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "synthetic_covid_dataset.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2400, 155)


,patient_id,age,sex,fever,cough,dyspnea,fatigue,headache,hypertension,diabetes,...,wavelet_014,wavelet_015,wavelet_016,wavelet_017,wavelet_018,wavelet_019,wavelet_020,ICU,mechanical_ventilation,death
0,P0001,57,0,0,1,0,1,0,0,0,...,-0.972052,-2.550070,-0.311566,0.868858,1.444569,-0.022637,0.180994,0,0,0
1,P0002,34,0,1,0,0,1,1,0,0,...,-0.875412,1.650275,1.353153,-0.679142,1.987041,0.120426,1.393416,0,0,0
2,P0003,65,0,1,1,0,0,0,1,1,...,0.509871,1.569078,0.068379,0.942279,0.389688,2.734910,-0.042336,1,1,1
3,P0004,68,0,0,1,0,0,1,0,0,...,-0.246149,-0.958745,0.140452,-1.069833,-0.288854,0.370459,0.482073,0,1,0
4,P0005,19,0,1,0,0,0,0,0,0,...,-1.255056,-1.511882,-1.616099,-0.182825,-0.631720,-2.478670,-0.721141,0,0,0


In [3]:

clinical_features = [
    "age", "sex", "fever", "cough", "dyspnea",
    "fatigue", "headache", "hypertension", "diabetes",
    "cardiovascular_disease", "copd", "chronic_liver_disease",
    "chronic_kidney_disease", "cancer"
]

lab_features = [
    "wbc", "neutrophil", "lymphocyte", "hemoglobin",
    "platelet", "d_dimer", "crp", "albumin", "alt", "ast",
    "bilirubin", "potassium", "sodium", "creatinine",
    "ck", "ldh", "hbdh"
]

radiomics_features = [
    col for col in df.columns
    if col.startswith((
        "firstorder_", "glcm_", "glszm_",
        "glrlm_", "ngtdm_", "wavelet_"
    ))
]

target_features = [
    "ICU", "mechanical_ventilation", "death"
]

print("Clinical features:", len(clinical_features))
print("Laboratory features:", len(lab_features))
print("Radiomics features:", len(radiomics_features))
print("Targets:", len(target_features))


Clinical features: 14
Laboratory features: 17
Radiomics features: 120
Targets: 3


In [4]:

categorical_candidates = [
    "sex", "fever", "cough", "dyspnea", "fatigue",
    "headache", "hypertension", "diabetes",
    "cardiovascular_disease", "copd",
    "chronic_liver_disease", "chronic_kidney_disease",
    "cancer"
]

for col in categorical_candidates:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))



sex:
sex
1    1375
0    1025
Name: count, dtype: int64

fever:
fever
1    1721
0     679
Name: count, dtype: int64

cough:
cough
1    1497
0     903
Name: count, dtype: int64

dyspnea:
dyspnea
0    1794
1     606
Name: count, dtype: int64

fatigue:
fatigue
0    1244
1    1156
Name: count, dtype: int64

headache:
headache
0    1784
1     616
Name: count, dtype: int64

hypertension:
hypertension
0    1703
1     697
Name: count, dtype: int64

diabetes:
diabetes
0    1980
1     420
Name: count, dtype: int64

cardiovascular_disease:
cardiovascular_disease
0    2114
1     286
Name: count, dtype: int64

copd:
copd
0    2248
1     152
Name: count, dtype: int64

chronic_liver_disease:
chronic_liver_disease
0    2261
1     139
Name: count, dtype: int64

chronic_kidney_disease:
chronic_kidney_disease
0    2227
1     173
Name: count, dtype: int64

cancer:
cancer
0    2287
1     113
Name: count, dtype: int64


In [5]:

all_features = clinical_features + lab_features + radiomics_features

# Confirm there are no duplicate feature names
assert len(all_features) == len(set(all_features))

# Check that all specified columns exist
missing_columns = [
    col for col in all_features + target_features
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

print("Total predictor features:", len(all_features))
print("Patient ID excluded:", "patient_id" not in all_features)


Total predictor features: 151
Patient ID excluded: True


In [6]:

from sklearn.model_selection import train_test_split

X = df[all_features].copy()
Y = df[target_features].copy()

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.30,
    random_state=42,
    stratify=Y["ICU"]
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nICU positive rate:")
print("Train:", round(Y_train["ICU"].mean(), 3))
print("Test: ", round(Y_test["ICU"].mean(), 3))


Training rows: 1680
Testing rows: 720

ICU positive rate:
Train: 0.293
Test:  0.293


In [7]:

from pandas.api.types import is_numeric_dtype

categorical_cols = [
    col for col in clinical_features
    if (
        not is_numeric_dtype(X_train[col])
        or X_train[col].nunique(dropna=True) <= 2
    )
]

numeric_cols = [
    col for col in all_features
    if col not in categorical_cols
]

print("Categorical columns:", categorical_cols)
print("Numerical columns:", len(numeric_cols))
print("Total predictors:", len(categorical_cols) + len(numeric_cols))


Categorical columns: ['sex', 'fever', 'cough', 'dyspnea', 'fatigue', 'headache', 'hypertension', 'diabetes', 'cardiovascular_disease', 'copd', 'chronic_liver_disease', 'chronic_kidney_disease', 'cancer']
Numerical columns: 138
Total predictors: 151


In [8]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols)
    ],
    remainder="drop"
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)
print("Any missing values remaining in training:",
      np.isnan(X_train_processed.toarray()).any()
      if hasattr(X_train_processed, "toarray")
      else np.isnan(X_train_processed).any())


Processed training shape: (1680, 164)
Processed testing shape: (720, 164)
Any missing values remaining in training: False


In [9]:

from pathlib import Path
import joblib
from scipy import sparse

OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def save_array(array, path):
    if sparse.issparse(array):
        sparse.save_npz(path.with_suffix(".npz"), array)
    else:
        np.save(path.with_suffix(".npy"), array)

save_array(
    X_train_processed,
    OUTPUT_DIR / "X_train_processed"
)
save_array(
    X_test_processed,
    OUTPUT_DIR / "X_test_processed"
)

Y_train.to_csv(OUTPUT_DIR / "Y_train.csv", index=False)
Y_test.to_csv(OUTPUT_DIR / "Y_test.csv", index=False)

joblib.dump(preprocessor, OUTPUT_DIR / "preprocessor.joblib")

print("Preprocessing outputs saved to:", OUTPUT_DIR)


Preprocessing outputs saved to: ../data/processed
